# Dataset 재구성

- Taxonomy 및 train/val/test split 재생성
- 원본 스크립트: `aws_icon_dataset_rebuild.sh`

## 개요
AWS 아이콘 taxonomy + train/val/test split 재생성

### 사용법
```bash
./aws_icon_dataset_rebuild.sh ./dataset/icons
```

### 인자
- `$1`: CSV 파일이 들어있는 디렉터리 (기본값: ./dataset/icons)

### 동작
기존 labels_coarse.csv / labels_fine.csv / train/val/test_fine.csv 기준으로 재계산을 수행합니다.


In [100]:
from __future__ import annotations

import os, json
from collections import Counter
from pathlib import Path
from pprint import pprint

import pandas as pd
from sklearn.model_selection import train_test_split

# ----- Config -----
DATA_DIR = os.environ["DATA_DIR"]
data_path = Path(DATA_DIR)

# ----- Helpers -----
def find_csv(root: Path, name: str):
    for p in root.rglob(name):
        return p
    return None

def ensure_data_dir(path_str: str | None = None) -> Path:
    """Resolve and validate DATA_DIR, mirroring the original shell logic."""
    base = path_str or os.environ.get("DATA_DIR") or "./dataset/icons"
    resolved = Path(base).expanduser().resolve()
    if not resolved.is_dir():
        raise FileNotFoundError(f"DATA_DIR '{resolved}' 디렉터리가 없습니다.")
    return resolved

data_dir = ensure_data_dir()
os.environ["DATA_DIR"] = str(data_dir)

print(f"[INFO] DATA_DIR = {data_dir}")
print("[INFO] 기존 labels_coarse.csv / labels_fine.csv / train/val/test_fine.csv 기준으로 재계산을 수행합니다.")


[INFO] DATA_DIR = /home/wsm/workspace/hit-archlens-project/dataset/icons
[INFO] 기존 labels_coarse.csv / labels_fine.csv / train/val/test_fine.csv 기준으로 재계산을 수행합니다.


In [101]:
labels_fine_path = find_csv(data_path, "labels_fine.csv")
labels_coarse_path = find_csv(data_path, "labels_coarse.csv")

print(f"[INFO] Using labels_fine.csv: {labels_fine_path}")
print(f"[INFO] Using labels_coarse.csv: {labels_coarse_path}")

print(f"[INFO] Load: {labels_fine_path}")
df_fine = pd.read_csv(labels_fine_path)

print(f"[INFO] Load: {labels_coarse_path}")
df_coarse = pd.read_csv(labels_coarse_path)

[INFO] Using labels_fine.csv: /home/wsm/workspace/hit-archlens-project/dataset/icons/labels_fine.csv
[INFO] Using labels_coarse.csv: /home/wsm/workspace/hit-archlens-project/dataset/icons/labels_coarse.csv
[INFO] Load: /home/wsm/workspace/hit-archlens-project/dataset/icons/labels_fine.csv
[INFO] Load: /home/wsm/workspace/hit-archlens-project/dataset/icons/labels_coarse.csv


### 1. Taxonomy 재구성 (coarse / fine)

In [102]:
# coarse taxonomy
coarse_counts = df_fine["coarse_class"].value_counts().sort_index()
df_tax_coarse = pd.DataFrame({
    "coarse_class": coarse_counts.index,
    "num_icons_fine": coarse_counts.values,
})

coarse_out = data_path / "taxonomy_coarse.csv"
df_tax_coarse.to_csv(coarse_out, index=False)
print(f"[OK] taxonomy_coarse.csv 저장: {coarse_out} (rows={len(df_tax_coarse)})")

# fine taxonomy (canonical_service_name + coarse + count)
fine_group = df_fine.groupby(
    ["coarse_class", "canonical_service_name"], as_index=False
).agg(num_icons=("file_path", "count"))

fine_group = fine_group.sort_values(
    ["coarse_class", "canonical_service_name"]
).reset_index(drop=True)

fine_out = data_path / "taxonomy_fine.csv"
fine_group.to_csv(fine_out, index=False)
print(f"[OK] taxonomy_fine.csv 저장: {fine_out} (rows={len(fine_group)})")

# stage1/2 class 리스트 텍스트 (라벨링/YOLO names용)
stage1_path = data_path / "stage1_classes.txt"
stage2_path = data_path / "stage2_classes.txt"

coarse_list = list(coarse_counts.index)
fine_list = list(fine_group["canonical_service_name"].unique())

stage1_path.write_text("\n".join(coarse_list), encoding="utf-8")
stage2_path.write_text("\n".join(fine_list), encoding="utf-8")

print(f"[OK] stage1_classes.txt 저장: {stage1_path} (coarse {len(coarse_list)}개)")
print(f"[OK] stage2_classes.txt 저장: {stage2_path} (fine {len(fine_list)}개)")


[OK] taxonomy_coarse.csv 저장: /home/wsm/workspace/hit-archlens-project/dataset/icons/taxonomy_coarse.csv (rows=19)
[OK] taxonomy_fine.csv 저장: /home/wsm/workspace/hit-archlens-project/dataset/icons/taxonomy_fine.csv (rows=64)
[OK] stage1_classes.txt 저장: /home/wsm/workspace/hit-archlens-project/dataset/icons/stage1_classes.txt (coarse 19개)
[OK] stage2_classes.txt 저장: /home/wsm/workspace/hit-archlens-project/dataset/icons/stage2_classes.txt (fine 64개)


In [103]:
pd.read_csv(coarse_out)

,coarse_class,num_icons_fine
0,AI & Machine Learning,41
1,Analytics,45
2,Application Integration,20
3,Blockchain,5
4,Business Applications,38
5,Compute,42
6,Containers & Orchestration,18
7,Database,71
8,DevOps & Developer Tools,7
9,IoT,39


In [104]:
pd.read_csv(fine_out)

,coarse_class,canonical_service_name,num_icons
0,AI & Machine Learning,amazon comprehend,8
1,AI & Machine Learning,amazon rekognition,6
2,AI & Machine Learning,amazon sagemaker,22
3,AI & Machine Learning,amazon textract,5
4,Analytics,amazon athena,5
...,...,...,...
59,Storage,amazon efs,9
60,Storage,amazon fsx,22
61,Storage,amazon s3,25
62,Storage,aws backup,21


In [105]:
stage1_class_list = stage1_path.read_text(encoding="utf-8").split("\n")
pprint(stage1_class_list, compact=True)

['AI & Machine Learning', 'Analytics', 'Application Integration', 'Blockchain',
 'Business Applications', 'Compute', 'Containers & Orchestration', 'Database',
 'DevOps & Developer Tools', 'IoT', 'Management & Governance',
 'Migration & Transfer', 'Monitoring & Logging', 'Networking', 'Quantum',
 'Robotics / AR-VR', 'Security & Identity', 'Serverless & Event-driven',
 'Storage']


In [106]:
stage2_class_list = stage2_path.read_text(encoding="utf-8").split("\n")
pprint(stage2_class_list, compact=True)

['amazon comprehend', 'amazon rekognition', 'amazon sagemaker',
 'amazon textract', 'amazon athena', 'amazon emr', 'amazon opensearch service',
 'amazon quicksight', 'aws glue', 'aws lake formation', 'amazon api gateway',
 'amazon mq', 'aws app mesh', 'amazon managed blockchain', 'amazon chime',
 'amazon connect', 'amazon workdocs', 'amazon workspaces', 'amazon ec2',
 'amazon lightsail', 'aws batch', 'aws elastic beanstalk', 'amazon ecs',
 'amazon eks', 'amazon aurora', 'amazon documentdb', 'amazon dynamodb',
 'amazon elasticache', 'amazon rds', 'amazon redshift', 'aws cloudformation',
 'aws iot analytics', 'aws iot core', 'aws iot greengrass', 'aws iot sitewise',
 'aws license manager', 'aws organizations', 'aws systems manager',
 'aws trusted advisor', 'aws database migration service', 'aws datasync',
 'aws snowball', 'aws transfer family', 'amazon cloudwatch', 'aws cloudtrail',
 'aws config', 'amazon route 53', 'amazon vpc', 'aws cloud map',
 'aws direct connect', 'aws transit gatew

### 2. train/val/test split 재생성 (stratified by coarse_class)
기본 비율: 0.7 / 0.15 / 0.15

In [107]:
N = len(df_fine)
train_ratio, val_ratio, test_ratio = 0.7, 0.15, 0.15
TARGET_VAL = int(round(N * val_ratio))
TARGET_TEST = int(round(N * test_ratio))

# --- 1) fine 클래스별로 최소 샘플 시드 확보 ---
val_seed_list: list[pd.DataFrame] = []
test_seed_list: list[pd.DataFrame] = []
remain_list: list[pd.DataFrame] = []

for _, g in df_fine.groupby("canonical_service_name"):
    g = g.sort_values("file_path")
    n = len(g)
    if n >= 3:  # 1장 val, 1장 test 시드
        val_seed_list.append(g.iloc[[0]])
        test_seed_list.append(g.iloc[[1]])
        remain_list.append(g.iloc[2:])
    elif n == 2:  # val 1장만 확보
        val_seed_list.append(g.iloc[[0]])
        remain_list.append(g.iloc[[1]])
    else:  # n == 1: 모두 train 후보로 남김
        remain_list.append(g)

val_seed = pd.concat(val_seed_list) if val_seed_list else df_fine.iloc[0:0]
test_seed = pd.concat(test_seed_list) if test_seed_list else df_fine.iloc[0:0]
remaining = pd.concat(remain_list) if remain_list else df_fine.iloc[0:0]


def stratify_or_none(df: pd.DataFrame):
    if len(df) < 2:
        return None
    counts = df["coarse_class"].value_counts()
    return df["coarse_class"] if counts.min() >= 2 else None

In [108]:
# --- 2) 남은 표본으로 목표 비율에 맞춰 분할 (coarse stratify) ---
val_need = max(TARGET_VAL - len(val_seed), 0)
test_need = max(TARGET_TEST - len(test_seed), 0)
pool_need = val_need + test_need

# 기본값
df_val_add = df_fine.iloc[0:0]
df_test_add = df_fine.iloc[0:0]
df_train_rest = remaining

if pool_need > 0 and len(remaining) > 0:
    pool_size = min(pool_need, len(remaining))
    # remaining에서 val/test 풀만큼만 추출하고 나머지는 train으로 남긴다.
    df_train_rest, valtest_pool = train_test_split(
        remaining,
        test_size=pool_size / len(remaining),
        stratify=stratify_or_none(remaining),
        random_state=42,
    )
    if len(valtest_pool) > 0:
        frac_val = val_need / pool_need if pool_need else 0.5
        df_val_add, df_test_add = train_test_split(
            valtest_pool,
            test_size=(1.0 - frac_val),
            stratify=stratify_or_none(valtest_pool),
            random_state=42,
        )

In [109]:
# --- 3) train/val/test 확정 ---
df_val = pd.concat([val_seed, df_val_add]).drop_duplicates(subset="file_path")
df_test = pd.concat([test_seed, df_test_add]).drop_duplicates(subset="file_path")
df_train = df_train_rest.copy()


# 정렬(옵션) - file_path 기준
for name in ["df_train", "df_val", "df_test"]:
    locals()[name].sort_values("file_path", inplace=True)
    locals()[name].reset_index(drop=True, inplace=True)

train_out = data_path / "train_fine.csv"
val_out = data_path / "val_fine.csv"
test_out = data_path / "test_fine.csv"

for path, df, label in [
    (train_out, locals()["df_train"], "train"),
    (val_out, locals()["df_val"], "val"),
    (test_out, locals()["df_test"], "test"),
]:
    df.to_csv(path, index=False)
    print(f"[OK] {label}_fine.csv 저장: {path} (rows={len(df)})")

# --- 4) fine 클래스 커버리지 출력 ---
expected = set(fine_group["canonical_service_name"])
for label, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    present = set(df["canonical_service_name"].unique())
    missing = expected - present
    print(
        f"[COVER] {label}: {len(present)}/{len(expected)} fine classes present; "
        f"missing={len(missing)}"
    )

[OK] train_fine.csv 저장: /home/wsm/workspace/hit-archlens-project/dataset/icons/train_fine.csv (rows=447)
[OK] val_fine.csv 저장: /home/wsm/workspace/hit-archlens-project/dataset/icons/val_fine.csv (rows=96)
[OK] test_fine.csv 저장: /home/wsm/workspace/hit-archlens-project/dataset/icons/test_fine.csv (rows=96)
[COVER] train: 64/64 fine classes present; missing=0
[COVER] val: 64/64 fine classes present; missing=0
[COVER] test: 64/64 fine classes present; missing=0


In [110]:
pd.read_csv(train_out)

,file_path,canonical_service_name,coarse_class,original_path
0,fine/amazon api gateway/Arch_Amazon-API-Gatewa...,amazon api gateway,Application Integration,Architecture-Service-Icons_02072025/Arch_Netwo...
1,fine/amazon athena/Arch_Amazon-Athena_48.png,amazon athena,Analytics,Architecture-Service-Icons_02072025/Arch_Analy...
2,fine/amazon athena/Arch_Amazon-Athena_64.png,amazon athena,Analytics,Architecture-Service-Icons_02072025/Arch_Analy...
3,fine/amazon athena/Res_Amazon-Athena_Data-Sour...,amazon athena,Analytics,Resource-Icons_02072025/Res_Analytics/Res_Amaz...
4,fine/amazon aurora/Arch_Amazon-Aurora_48.png,amazon aurora,Database,Architecture-Service-Icons_02072025/Arch_Datab...
...,...,...,...,...
442,fine/aws waf/Res_AWS-WAF_Rule_48.png,aws waf,Security & Identity,Resource-Icons_02072025/Res_Security-Identity-...
443,fine/elastic load balancing/Arch_Elastic-Load-...,elastic load balancing,Networking,Architecture-Service-Icons_02072025/Arch_Netwo...
444,fine/elastic load balancing/Arch_Elastic-Load-...,elastic load balancing,Networking,Architecture-Service-Icons_02072025/Arch_Netwo...
445,fine/elastic load balancing/Res_Elastic-Load-B...,elastic load balancing,Networking,Resource-Icons_02072025/Res_Networking-Content...


### 3. split 통계 JSON 생성 (train_val_test_split.json)

In [111]:
def stats_for(df):
    coarse = df["coarse_class"].value_counts().to_dict()
    fine = df["canonical_service_name"].value_counts().to_dict()
    return {
        "count": int(len(df)),
        "coarse_distribution": {k: int(v) for k, v in coarse.items()},
        "fine_distribution": {k: int(v) for k, v in fine.items()},
    }

split_stats = {
    "train": stats_for(df_train),
    "val": stats_for(df_val),
    "test": stats_for(df_test),
}

json_out = data_path / "train_val_test_split.json"
json_out.write_text(json.dumps(split_stats, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"[OK] train_val_test_split.json 저장: {json_out}")

print("[DONE] taxonomy + split + stats 재생성 완료.")

[OK] train_val_test_split.json 저장: /home/wsm/workspace/hit-archlens-project/dataset/icons/train_val_test_split.json
[DONE] taxonomy + split + stats 재생성 완료.


In [112]:
pprint(split_stats, compact=True)

{'test': {'coarse_distribution': {'AI & Machine Learning': 6,
                                  'Analytics': 7,
                                  'Application Integration': 3,
                                  'Blockchain': 1,
                                  'Business Applications': 6,
                                  'Compute': 4,
                                  'Containers & Orchestration': 3,
                                  'Database': 11,
                                  'DevOps & Developer Tools': 2,
                                  'IoT': 8,
                                  'Management & Governance': 7,
                                  'Migration & Transfer': 4,
                                  'Monitoring & Logging': 6,
                                  'Networking': 7,
                                  'Quantum': 3,
                                  'Robotics / AR-VR': 1,
                                  'Security & Identity': 3,
                                  '